<a href="https://colab.research.google.com/github/DevRodriguesx/CURSO_PYTHON/blob/main/catraca_facial_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Catraca de reconhecimento facial (tudo ao vivo, sem enviar fotos)

Neste notebook **não há upload de fotos**. Tudo é feito na hora, pela webcam:

1. **Cadastro:** para cada pessoa você digita o nome e a câmera captura **30 fotos** (com instruções na tela: de frente,
   virar para os lados, levantar o queixo, sorrir...). Repita para cada pessoa e digite **FIM** quando acabar.
2. **Treino:** uma ResNet pré-treinada recebe uma camada final de classificação (uma classe por pessoa) e passa por fine-tuning.
3. **Catraca:** a câmera abre como uma catraca de acesso.
   - Pessoa cadastrada: **quadrado verde**, nome e **ACESSO LIBERADO**.
   - Pessoa não cadastrada: **quadrado vermelho** e **PRECISA CADASTRAR**. Aparecem botões na própria catraca: **Cadastrar
     esta pessoa** (digite o nome; o modelo é **retreinado com todas as pessoas, as antigas e a nova**, e a catraca volta sozinha)
     ou **Ignorar**.

**Como usar:** `Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4` (opcional, deixa o treino bem mais rápido)
e rode as células **na ordem**. Permita o acesso à câmera quando o navegador pedir.

> Use apenas rostos de pessoas que concordaram em participar. As fotos ficam só nesta sessão do Colab e somem quando ela acaba.

## 1. Ambiente

In [ ]:
import os, io, re, json, time, shutil, random, unicodedata
from base64 import b64decode, b64encode
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display, Javascript
from google.colab import files
from google.colab.output import eval_js
from PIL import Image
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

print("OpenCV:", cv2.__version__, "| Torch:", torch.__version__)
assert hasattr(cv2, "CascadeClassifier"), (
    'OpenCV 5 detectado (sem CascadeClassifier). Rode: !pip install "opencv-python<5" '
    "e depois Ambiente de execução > Reiniciar sessão."
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE, "(GPU acelera bastante o treino)" if DEVICE == "cuda" else "(CPU: funciona, mas o treino é mais lento)")

SAIDA = "/content/dataset"              # rostos recortados, uma pasta por pessoa
ARQ_MODELO = "/content/modelo_faces.pt"
COMECAR_DO_ZERO = True                  # True: apaga cadastros antigos desta sessão ao rodar esta célula

if COMECAR_DO_ZERO:
    shutil.rmtree(SAIDA, ignore_errors=True)
os.makedirs(SAIDA, exist_ok=True)

## 2. Funções base (detecção do rosto, recorte, modelo e imagens)

In [ ]:
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]  # estatísticas do ImageNet (usadas no pré-treino da ResNet)
STD = [0.229, 0.224, 0.225]


def limpar_nome(nome):
    # 'José Pedro' -> 'jose_pedro' (sem acento, minúsculo, sem espaço)
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    nome = "_".join(nome.strip().lower().split())
    return re.sub(r"[^a-z0-9_]", "", nome)      # só letras, números e _ (seguro para pasta e para a tela)


def bonito(nome):
    return nome.replace("_", " ").title()


def agora():
    try:
        from zoneinfo import ZoneInfo
        return datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%H:%M:%S")
    except Exception:
        return datetime.now().strftime("%H:%M:%S")


detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


def largest_face(frame_bgr):
    # retorna (x, y, w, h) do maior rosto da imagem, ou None
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    faces = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))
    if len(faces) == 0:
        return None
    return max(faces, key=lambda f: f[2] * f[3])


def crop_with_margin(frame_bgr, box, margin=0.2):
    # recorta o rosto com margem extra e redimensiona para 224x224
    x, y, w, h = [int(v) for v in box]
    mx, my = int(w * margin), int(h * margin)
    x0, y0 = max(0, x - mx), max(0, y - my)
    x1, y1 = min(frame_bgr.shape[1], x + w + mx), min(frame_bgr.shape[0], y + h + my)
    return cv2.resize(frame_bgr[y0:y1, x0:x1], (IMG_SIZE, IMG_SIZE))


def build_model(arch, num_classes, pretrained=True):
    # ResNet pré-treinada com a camada final trocada por uma camada de classificação
    # (1 neurônio de saída por pessoa cadastrada)
    if arch == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
    else:
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
    for p in model.parameters():      # congela o backbone
        p.requires_grad = False
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))
    return model


TF_TRAIN = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
TF_VAL = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
tf_pred = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

modelo, CLASSES = None, []   # preenchidos pelo treino


def prever(img_bgr, box):
    # retorna (nome_da_classe, confiança) para o rosto dentro de 'box'
    rosto_rgb = cv2.cvtColor(crop_with_margin(img_bgr, box), cv2.COLOR_BGR2RGB)
    tensor = tf_pred(rosto_rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(modelo(tensor), dim=1)[0]
    conf, idx = probs.max(0)
    return CLASSES[idx.item()], conf.item()


def js_para_imagem(data_url):
    arr = np.frombuffer(b64decode(data_url.split(",")[1]), np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)


def salvar_rosto(img_bgr, classe):
    # detecta o rosto, recorta e salva em dataset/<classe>/. Retorna o caminho salvo (ou None).
    box = largest_face(img_bgr)
    if box is None:
        return None
    destino = Path(SAIDA) / classe
    destino.mkdir(parents=True, exist_ok=True)
    caminho = destino / f"img_{len(list(destino.glob('*.jpg'))):04d}.jpg"
    cv2.imwrite(str(caminho), crop_with_margin(img_bgr, box))
    return caminho


def mostrar_previa(caminhos, titulo):
    if not caminhos:
        return
    idx = np.linspace(0, len(caminhos) - 1, min(8, len(caminhos))).astype(int)
    fig, eixos = plt.subplots(1, len(idx), figsize=(1.8 * len(idx), 2.2), squeeze=False)
    for ax, i in zip(eixos[0], idx):
        ax.imshow(cv2.cvtColor(cv2.imread(str(caminhos[i])), cv2.COLOR_BGR2RGB))
        ax.axis("off")
    fig.suptitle(titulo, fontsize=11)
    plt.tight_layout()
    plt.show()


def mostrar_imagem(img_bgr, titulo, cor="red"):
    plt.figure(figsize=(6, 5))
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(titulo, color=cor, fontsize=14)
    plt.axis("off")
    plt.show()


def contar_dataset():
    print("Pessoas cadastradas:")
    pessoas = sorted(os.listdir(SAIDA)) if os.path.isdir(SAIDA) else []
    if not pessoas:
        print("  (ninguém ainda)")
    for p in pessoas:
        n = len(list(Path(SAIDA, p).glob("*.jpg")))
        aviso = "  <-- poucas fotos, o reconhecimento pode falhar" if n < 15 else ""
        print(f"  {bonito(p)}: {n} fotos{aviso}")
    return pessoas


print("Funções carregadas.")

## 3. Captura pela webcam (30 fotos com instruções na tela)

A câmera abre com um botão **Iniciar captura**. Depois de uma contagem de 3 segundos, o notebook tira as fotos em sequência
e mostra instruções (de frente, virar para os lados, queixo para cima e para baixo, sorrir). Siga as instruções:
**variedade de ângulos e expressões é o que mais melhora o reconhecimento**.

In [ ]:
JS_CAPTURA = '''
async function capturar(qtd, intervaloMs, qualidade, instrucoes) {
  const div = document.createElement('div');
  const botao = document.createElement('button');
  botao.textContent = 'Iniciar captura';
  const msg = document.createElement('div');
  msg.style.font = 'bold 20px sans-serif';
  msg.style.margin = '8px 0';
  const video = document.createElement('video');
  video.style.display = 'block';
  div.appendChild(botao);
  div.appendChild(msg);
  div.appendChild(video);
  document.body.appendChild(div);
  const stream = await navigator.mediaDevices.getUserMedia({video: true});
  video.srcObject = stream;
  await video.play();
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  await new Promise((resolve) => botao.onclick = resolve);
  botao.disabled = true;
  for (let c = 3; c > 0; c--) {
    msg.textContent = 'Posicione o rosto... ' + c;
    await new Promise((r) => setTimeout(r, 1000));
  }
  const canvas = document.createElement('canvas');
  canvas.width = video.videoWidth;
  canvas.height = video.videoHeight;
  const ctx = canvas.getContext('2d');
  const fotos = [];
  for (let i = 0; i < qtd; i++) {
    const instr = instrucoes[Math.floor(i * instrucoes.length / qtd)];
    msg.textContent = 'Foto ' + (i + 1) + ' de ' + qtd + ' - ' + instr;
    await new Promise((r) => setTimeout(r, intervaloMs));
    ctx.drawImage(video, 0, 0);
    fotos.push(canvas.toDataURL('image/jpeg', qualidade));
  }
  stream.getVideoTracks()[0].stop();
  div.remove();
  return fotos;
}
'''

INSTRUCOES = [
    "olhe de frente",
    "vire um pouco o rosto para a ESQUERDA",
    "vire um pouco o rosto para a DIREITA",
    "levante um pouco o queixo",
    "abaixe um pouco o queixo",
    "sorria",
]


def cadastrar_pessoa(nome, qtd=30, intervalo=0.8):
    classe = limpar_nome(nome)
    display(Javascript(JS_CAPTURA))
    chamada = f"capturar({int(qtd)}, {int(intervalo * 1000)}, 0.9, {json.dumps(INSTRUCOES)})"
    try:
        fotos = eval_js(chamada, timeout_sec=300)
    except TypeError:                      # versões antigas do Colab sem timeout_sec
        fotos = eval_js(chamada)
    salvas = []
    for dados in fotos or []:
        img = js_para_imagem(dados)
        if img is None:
            continue
        caminho = salvar_rosto(img, classe)
        if caminho is not None:
            salvas.append(caminho)
    print(f"{bonito(classe)}: {len(salvas)} de {qtd} fotos tinham rosto detectado e foram salvas.")
    if len(salvas) < 15:
        print("Poucas fotos válidas. Fique de frente para a câmera, bem iluminado, sem se afastar demais.")
    mostrar_previa(salvas, f"Amostra das fotos de {bonito(classe)}")
    return len(salvas)


print("Captura pronta.")

## 4. Cadastrar as pessoas (repete até você digitar FIM)

Rode a célula. Ela pergunta o nome, abre a câmera e captura as fotos; depois pergunta a próxima pessoa.
**Digite `FIM`** quando não houver mais ninguém para cadastrar. É preciso pelo menos **2 pessoas**.

Se digitar um nome que já existe, as novas fotos são **somadas** às anteriores.
Use nomes sem acento (por exemplo `Jose Pedro`).

In [ ]:
QTD_FOTOS = 30   # fotos por pessoa

print("CADASTRO DE PESSOAS")
while True:
    nome = input("\nNome da próxima pessoa (ou digite FIM para terminar): ").strip()
    if nome.lower() == "fim":
        if len([p for p in os.listdir(SAIDA) if any(Path(SAIDA, p).glob("*.jpg"))]) < 2:
            print("Cadastre pelo menos 2 pessoas antes de terminar.")
            continue
        break
    if not limpar_nome(nome):
        print("Nome inválido, tente de novo.")
        continue
    if (Path(SAIDA) / limpar_nome(nome)).is_dir():
        print(f"{bonito(limpar_nome(nome))} já está cadastrada: as novas fotos serão somadas.")
    print(f"Prepare {bonito(limpar_nome(nome))} na frente da câmera e clique em 'Iniciar captura'.")
    cadastrar_pessoa(nome, QTD_FOTOS)

print("\nCadastro encerrado.")
contar_dataset()

## 5. Treinar o modelo (fine-tuning da ResNet)

Estratégia em 2 fases:
1. Treina só a nova camada de classificação (backbone congelado).
2. Descongela o último bloco (`layer4`) e ajusta com learning rate menor.

Esta função também será usada para **retreinar** quando você cadastrar alguém novo pela catraca.

In [ ]:
def stratified_split(targets, val_frac, seed=42):
    rng = random.Random(seed)
    por_classe = defaultdict(list)
    for i, t in enumerate(targets):
        por_classe[t].append(i)
    treino, val = [], []
    for idx in por_classe.values():
        rng.shuffle(idx)
        n_val = max(1, int(len(idx) * val_frac))
        val += idx[:n_val]
        treino += idx[n_val:]
    return treino, val


def run_epoch(model, loader, criterion, optimizer=None):
    treinando = optimizer is not None
    model.train(treinando)
    total_loss, acertos, n = 0.0, 0, 0
    with torch.set_grad_enabled(treinando):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)
            if treinando:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            acertos += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, acertos / n


def treinar(arch="resnet18", epocas_cabeca=8, epocas_ft=12, batch=16, val_frac=0.2, verbose=True):
    global modelo, CLASSES
    base = datasets.ImageFolder(SAIDA)
    classes = base.classes
    if len(classes) < 2:
        raise RuntimeError("Cadastre pelo menos 2 pessoas antes de treinar.")
    contagem = {bonito(c): base.targets.count(i) for i, c in enumerate(classes)}
    poucas = [nome for nome, n in contagem.items() if n < 6]
    if poucas:
        raise RuntimeError(f"Poucas fotos para: {poucas}. Cadastre essas pessoas de novo (mínimo de 6 fotos, ideal 30).")
    idx_train, idx_val = stratified_split(base.targets, val_frac)
    ds_train = Subset(datasets.ImageFolder(SAIDA, transform=TF_TRAIN), idx_train)
    ds_val = Subset(datasets.ImageFolder(SAIDA, transform=TF_VAL), idx_val)
    dl_train = DataLoader(ds_train, batch_size=batch, shuffle=True)
    dl_val = DataLoader(ds_val, batch_size=batch)
    print(f"Treinando com TODAS as pessoas cadastradas (fotos por pessoa: {contagem}) | "
          f"treino: {len(ds_train)} | validação: {len(ds_val)}")

    model = build_model(arch, len(classes)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    melhor, melhor_estado = -1.0, None

    def fase(nome_fase, epocas, opt):
        nonlocal melhor, melhor_estado
        for ep in range(1, epocas + 1):
            _, ta = run_epoch(model, dl_train, criterion, opt)
            vl, va = run_epoch(model, dl_val, criterion)
            if verbose:
                print(f"[{nome_fase}] ép {ep:02d} | treino {ta:.2%} | val {va:.2%} (loss {vl:.3f})")
            if va >= melhor:
                melhor = va
                melhor_estado = {k: v.detach().clone() for k, v in model.state_dict().items()}

    # ---- Fase 1: só a camada final ----
    fase("Fase 1", epocas_cabeca, torch.optim.Adam(model.fc.parameters(), lr=1e-3))

    # ---- Fase 2: fine-tuning do último bloco ----
    for p in model.layer4.parameters():
        p.requires_grad = True
    fase("Fase 2", epocas_ft, torch.optim.Adam([
        {"params": model.layer4.parameters(), "lr": 1e-4},
        {"params": model.fc.parameters(), "lr": 5e-4},
    ]))

    model.load_state_dict(melhor_estado)      # fica com o melhor modelo da validação
    model.eval()
    modelo, CLASSES = model, classes
    torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
                "classes": classes, "arch": arch}, ARQ_MODELO)
    print(f"Treino concluído. Melhor acurácia de validação: {melhor:.2%} (modelo salvo em {ARQ_MODELO})")
    return melhor


print("Treino pronto.")

In [ ]:
treinar()

## 6. A catraca (reconhecimento ao vivo)

A câmera abre como uma catraca de acesso, olhando **uma pessoa por vez** (o rosto maior da imagem):

- 🟩 **Verde + nome + ACESSO LIBERADO:** pessoa reconhecida.
- 🟥 **Vermelho + PRECISA CADASTRAR:** pessoa não cadastrada. Aparecem, **abaixo do vídeo**, um campo de nome e os botões
  **Cadastrar esta pessoa** e **Ignorar**.
  - **Cadastrar:** digite o nome e clique no botão. A câmera captura 30 fotos (com instruções na tela), e o notebook
    **retreina o modelo do zero com TODAS as pessoas** (as que já estavam + a nova). Quando o treino acaba, a catraca volta
    sozinha e a pessoa nova já é reconhecida. O texto no notebook mostra a lista de pessoas usadas no treino.
  - **Ignorar:** a catraca segue sem cadastrar.
- 🟨 **Amarelo:** analisando.

Para **encerrar**, clique no botão **Encerrar catraca**. (Clicar no vídeo não faz nada.)

Para evitar decisões por engano, o acesso só muda depois de `CONFIRMAR` quadros seguidos com o mesmo resultado, e a confiança precisa
ser de pelo menos `LIMIAR`.

> Limitação importante: o modelo só conhece quem foi cadastrado e **sempre escolhe a pessoa mais parecida**. Um desconhecido muito
> parecido com alguém cadastrado pode ser liberado por engano. Se isso acontecer nos testes, aumente o `LIMIAR` (por exemplo 0.90).

In [ ]:
LIMIAR = 0.80      # confiança mínima para liberar o acesso
CONFIRMAR = 3      # quadros seguidos iguais para confirmar a decisão
DURACAO = 1800     # segundos máximos por sessão da catraca (limite de segurança)
MIN_FOTOS = 15     # mínimo de fotos válidas para aceitar um cadastro novo

VERDE, VERMELHO, AMARELO = (0, 170, 0, 255), (220, 0, 0, 255), (230, 160, 0, 255)
registro = []      # histórico de acessos liberados: (hora, nome)

JS_VIDEO = '''
var video, div = null, stream, captureCanvas, imgElement, labelElement;
var acoes, campoNome, btnCadastrar, btnIgnorar, btnEncerrar;
var decisaoResolve = null;
var pendingResolve = null;
var shutdown = false;

function ajustarAltura() {
  try { google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true); } catch (e) {}
}

function removeDom() {
  if (stream) { stream.getVideoTracks()[0].stop(); }
  if (div) { div.remove(); }
  video = null; div = null; stream = null; imgElement = null; captureCanvas = null; labelElement = null;
  acoes = null; decisaoResolve = null;
}

function onAnimationFrame() {
  if (!shutdown) { window.requestAnimationFrame(onAnimationFrame); }
  if (pendingResolve) {
    var result = "";
    if (!shutdown) {
      captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
      result = captureCanvas.toDataURL('image/jpeg', 0.8);
    }
    var lp = pendingResolve;
    pendingResolve = null;
    lp(result);
  }
}

async function createDom() {
  if (div !== null) { return stream; }
  div = document.createElement('div');
  div.style.border = '2px solid black';
  div.style.padding = '3px';
  div.style.width = '100%';
  div.style.maxWidth = '640px';
  document.body.appendChild(div);

  const topo = document.createElement('div');
  topo.innerHTML = '<span>Catraca: </span>';
  labelElement = document.createElement('span');
  labelElement.innerText = 'iniciando...';
  labelElement.style.fontWeight = 'bold';
  topo.appendChild(labelElement);
  div.appendChild(topo);

  video = document.createElement('video');
  video.style.display = 'block';
  video.width = div.clientWidth - 6;
  video.setAttribute('playsinline', '');
  stream = await navigator.mediaDevices.getUserMedia({video: true});
  div.appendChild(video);

  imgElement = document.createElement('img');
  imgElement.style.position = 'absolute';
  imgElement.style.zIndex = 1;
  div.appendChild(imgElement);

  acoes = document.createElement('div');
  acoes.style.display = 'none';
  acoes.style.margin = '8px 0';
  campoNome = document.createElement('input');
  campoNome.type = 'text';
  campoNome.placeholder = 'Nome (sem acento)';
  campoNome.style.fontSize = '16px';
  campoNome.style.marginRight = '6px';
  btnCadastrar = document.createElement('button');
  btnCadastrar.textContent = 'Cadastrar esta pessoa';
  btnIgnorar = document.createElement('button');
  btnIgnorar.textContent = 'Ignorar';
  btnIgnorar.style.marginLeft = '6px';
  acoes.appendChild(campoNome);
  acoes.appendChild(btnCadastrar);
  acoes.appendChild(btnIgnorar);
  div.appendChild(acoes);

  btnEncerrar = document.createElement('button');
  btnEncerrar.textContent = 'Encerrar catraca';
  btnEncerrar.style.marginTop = '6px';
  btnEncerrar.onclick = () => {
    shutdown = true;
    if (decisaoResolve) { decisaoResolve({acao: 'encerrar', nome: ''}); decisaoResolve = null; }
  };
  div.appendChild(btnEncerrar);

  video.srcObject = stream;
  await video.play();

  captureCanvas = document.createElement('canvas');
  captureCanvas.width = 640;
  captureCanvas.height = 480;
  window.requestAnimationFrame(onAnimationFrame);
  ajustarAltura();
  return stream;
}

async function stream_frame(label, imgData) {
  if (shutdown) { removeDom(); shutdown = false; return ''; }
  stream = (!stream) ? await createDom() : stream;
  if (label != "") { labelElement.innerHTML = label; }
  if (imgData != "") {
    var videoRect = video.getClientRects()[0];
    imgElement.style.top = videoRect.top + "px";
    imgElement.style.left = videoRect.left + "px";
    imgElement.style.width = videoRect.width + "px";
    imgElement.style.height = videoRect.height + "px";
    imgElement.style.display = 'block';
    imgElement.src = imgData;
  }
  var result = await new Promise(function(resolve, reject) { pendingResolve = resolve; });
  return {'img': result};
}

async function esperar_decisao() {
  // mostra o campo de nome e os botões; devolve {acao: 'cadastrar'|'ignorar'|'encerrar', nome}
  campoNome.value = '';
  campoNome.style.border = '';
  acoes.style.display = 'block';
  ajustarAltura();
  campoNome.focus();
  const r = await new Promise((resolve) => {
    decisaoResolve = resolve;
    btnCadastrar.onclick = () => {
      const n = campoNome.value.trim();
      if (n === '') { campoNome.style.border = '2px solid red'; campoNome.focus(); return; }
      resolve({acao: 'cadastrar', nome: n});
    };
    btnIgnorar.onclick = () => resolve({acao: 'ignorar', nome: ''});
    campoNome.onkeydown = (e) => { if (e.key === 'Enter') { btnCadastrar.click(); } };
  });
  decisaoResolve = null;
  if (acoes) { acoes.style.display = 'none'; }
  return r;
}

async function capturar_do_video(qtd, intervaloMs, instrucoes) {
  // tira 'qtd' fotos do próprio vídeo da catraca (a câmera continua aberta)
  imgElement.style.display = 'none';
  const canvas = document.createElement('canvas');
  canvas.width = 640;
  canvas.height = 480;
  const ctx = canvas.getContext('2d');
  const fotos = [];
  for (let c = 3; c > 0; c--) {
    labelElement.innerHTML = "<span style='color:#06c;font-size:20px'>Posicione o rosto... " + c + "</span>";
    await new Promise((r) => setTimeout(r, 1000));
  }
  for (let i = 0; i < qtd; i++) {
    const instr = instrucoes[Math.floor(i * instrucoes.length / qtd)];
    labelElement.innerHTML = "<span style='color:#06c;font-size:20px'>Foto " + (i + 1) + " de " + qtd + " - " + instr + "</span>";
    await new Promise((r) => setTimeout(r, intervaloMs));
    ctx.drawImage(video, 0, 0, 640, 480);
    fotos.push(canvas.toDataURL('image/jpeg', 0.9));
  }
  return fotos;
}

async function parar() {
  shutdown = true;
  return await stream_frame("", "");
}
'''


def overlay_para_data_url(overlay_rgba):
    buf = io.BytesIO()
    Image.fromarray(overlay_rgba).save(buf, format="png")
    return "data:image/png;base64," + b64encode(buf.getvalue()).decode()


def html_status(texto, cor):
    return f"<span style='color:{cor};font-size:20px'>{texto}</span>"


def desenhar(overlay, box, cor, titulo, subtitulo):
    # caixa colorida + faixa com o nome (em cima) e o status (embaixo), numa camada transparente
    x, y, w, h = [int(v) for v in box]
    cv2.rectangle(overlay, (x, y), (x + w, y + h), cor, 4)

    def faixa(texto, base_y):
        (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
        x0 = min(max(0, x), max(0, 640 - tw - 10))
        y0 = min(max(th + 12, base_y), 470)
        cv2.rectangle(overlay, (x0, y0 - th - 8), (x0 + tw + 10, y0 + 6), cor, -1)
        cv2.putText(overlay, texto, (x0 + 5, y0), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255, 255), 2)

    faixa(titulo, y - 10)
    faixa(subtitulo, y + h + 34)


def js(expr, timeout=900):
    # executa JavaScript na página e devolve o resultado (com tempo limite maior para esperas longas)
    try:
        return eval_js(expr, timeout_sec=timeout)
    except TypeError:                       # versões antigas do Colab sem timeout_sec
        return eval_js(expr)


def mostrar_msg(texto, cor):
    # troca o texto de status da catraca (espera 1 quadro, então a câmera segue funcionando)
    js(f"stream_frame({json.dumps(html_status(texto, cor))}, '')", 60)


def salvar_fotos(fotos, classe):
    caminhos = []
    for dados in fotos or []:
        img = js_para_imagem(dados)
        if img is None:
            continue
        caminho = salvar_rosto(img, classe)
        if caminho is not None:
            caminhos.append(caminho)
    return caminhos


def tratar_desconhecido():
    # Mostra Cadastrar/Ignorar e executa a escolha. Retorna ("continuar" | "encerrar", segundos_de_carencia).
    criadas = set()          # pessoas criadas neste atendimento (para limpar se o usuário desistir)
    while True:
        try:
            r = js("esperar_decisao()") or {}
        except Exception as erro:
            print("Não consegui receber a resposta da tela:", erro)
            r = {"acao": "ignorar"}
        acao = r.get("acao", "ignorar")

        if acao == "encerrar":
            return "encerrar", 0

        if acao != "cadastrar":                                      # ignorar
            for classe in criadas:                                   # remove cadastros incompletos
                pasta = Path(SAIDA) / classe
                if len(list(pasta.glob("*.jpg"))) < MIN_FOTOS:
                    shutil.rmtree(pasta, ignore_errors=True)
            return "continuar", 6

        classe = limpar_nome(r.get("nome", ""))
        if not classe:
            mostrar_msg("Nome inválido. Use letras e números, sem acento.", "#d00")
            continue
        ja_existia = (Path(SAIDA) / classe).is_dir()

        # 1) captura 30 fotos da pessoa que está na frente da catraca
        fotos = js(f"capturar_do_video({QTD_FOTOS}, 800, {json.dumps(INSTRUCOES)})", 300)
        salvas = salvar_fotos(fotos, classe)
        if not ja_existia and salvas:
            criadas.add(classe)
        total = len(list((Path(SAIDA) / classe).glob("*.jpg")))
        print(f"[cadastro] {bonito(classe)}: {len(salvas)} de {QTD_FOTOS} fotos com rosto detectado (total da pessoa: {total}).")
        if total < MIN_FOTOS:
            mostrar_msg(f"Só {total} fotos válidas (mínimo {MIN_FOTOS}). Fique de frente e bem iluminado e tente de novo.", "#d00")
            continue                                                 # mostra os botões de novo; as fotos se somam

        # 2) retreina com TODAS as pessoas (antigas + nova)
        mostrar_msg(f"Treinando o modelo com {bonito(classe)}... aguarde (a câmera fica parada nesse tempo)", "#06c")
        antes = set(CLASSES)
        treinar(verbose=False)
        faltando = (antes | {classe}) - set(CLASSES)
        if faltando:
            print("ATENÇÃO: estas pessoas NÃO entraram no modelo:", sorted(faltando))
        print(f"[cadastro] Modelo atualizado. Pessoas reconhecidas agora: {[bonito(c) for c in CLASSES]}")
        return "continuar", 4


def catraca():
    if modelo is None:
        raise RuntimeError("Treine o modelo antes (seção 5).")
    display(Javascript(JS_VIDEO))
    inicio = time.time()
    libera_apos = inicio                    # até esse horário a catraca só mostra "Aguarde..."
    bbox, status = "", html_status("Iniciando câmera...", "#555")
    hist, atual = [], None
    try:
        while time.time() - inicio < DURACAO:
            resp = eval_js(f"stream_frame({json.dumps(status)}, {json.dumps(bbox)})")
            if not resp or not resp.get("img"):
                break                                   # clicou em 'Encerrar catraca'
            frame = js_para_imagem(resp["img"])
            overlay = np.zeros((480, 640, 4), dtype=np.uint8)    # camada transparente sobre o vídeo
            box = largest_face(frame)

            if box is None:                             # ninguém na frente da catraca
                hist, atual = [], None
                status = html_status("Aguardando pessoa...", "#555")
                bbox = overlay_para_data_url(overlay)
                continue

            if time.time() < libera_apos:               # período de carência (após ignorar/cadastrar)
                desenhar(overlay, box, AMARELO, "Aguarde...", "reiniciando")
                status = html_status("Reiniciando...", "#b07800")
                bbox = overlay_para_data_url(overlay)
                continue

            nome, conf = prever(frame, box)
            hist = (hist + [nome if conf >= LIMIAR else None])[-CONFIRMAR:]
            confirmado = len(hist) == CONFIRMAR and len(set(hist)) == 1

            if not confirmado:
                desenhar(overlay, box, AMARELO, "Analisando...", "aguarde")
                status = html_status("Analisando...", "#b07800")
            elif hist[0] is not None:                   # pessoa cadastrada
                nome_ok = bonito(hist[0])
                desenhar(overlay, box, VERDE, nome_ok, "ACESSO LIBERADO")
                status = html_status(f"ACESSO LIBERADO - {nome_ok}", "#0a0")
                if atual != hist[0]:
                    atual = hist[0]
                    registro.append((agora(), nome_ok))
            else:                                       # pessoa NÃO cadastrada
                desenhar(overlay, box, VERMELHO, "Nao cadastrado", "PRECISA CADASTRAR")
                status = html_status("PRECISA CADASTRAR - use os botões abaixo do vídeo", "#d00")
                bbox = overlay_para_data_url(overlay)
                eval_js(f"stream_frame({json.dumps(status)}, {json.dumps(bbox)})")   # mostra o vermelho na tela
                acao, carencia = tratar_desconhecido()
                if acao == "encerrar":
                    break
                hist, atual = [], None
                libera_apos = time.time() + carencia
                status = html_status("Reiniciando...", "#b07800")
                bbox = overlay_para_data_url(np.zeros((480, 640, 4), dtype=np.uint8))
                continue
            bbox = overlay_para_data_url(overlay)
        else:
            print("Tempo máximo da sessão atingido. Rode a célula de novo para reabrir a catraca.")
    finally:
        try:
            eval_js("parar()")                          # desliga a câmera e remove o vídeo da tela
        except Exception:
            pass


print("Catraca pronta.")

In [ ]:
registro.clear()
catraca()

print("\nCatraca encerrada.")
print("Pessoas reconhecidas pelo modelo:", [bonito(c) for c in CLASSES])
print("Acessos liberados nesta sessão:")
for hora, nome in registro:
    print(f"  {hora}  {nome}")
if not registro:
    print("  (nenhum)")

## 7. (Opcional) Baixar o modelo treinado
O arquivo `modelo_faces.pt` guarda o modelo e a lista de pessoas. As fotos ficam só nesta sessão e somem quando ela acaba.

In [ ]:
files.download(ARQ_MODELO)